# 09 · Baseten Muse Spark Open → optional SAM 3.1 → Baseten Muse Spark Open

Run this in Colab to compare the hackathon Baseten model with the deployed PetSafe Plants flow. **All Baseten calls use exactly `meta-models/muse-spark-open-1026`**, the model ID supplied for the hackathon key. The scene prompt, allowed toxicity lookup keys, JSON fields, SAM decision, and region filtering are snapshots of the local project at notebook creation. This notebook does not call the deployed app or perform verified toxicity lookup.

Add `BASETEN_API_KEY` to **Colab Secrets** and grant notebook access. SAM is hosted by **Meta Model API**, so optional segmentation also needs a separate `MODEL_API_KEY` secret. Keys are read into environment variables in the runtime and are never saved in notebook outputs. Upload one photograph; Baseten receives the original and, if SAM runs, up to six selected transparent crops. These calls may incur charges.

References: [Meta's Baseten deployment guide](https://dev.meta.ai/docs/muse-spark-open/deploy), [SAM client libraries](https://dev.meta.ai/docs/sam/client-libraries). The public guide shows a general model ID; this notebook intentionally uses the hackathon-specific ID above.

## 1. Install dependencies and configure credentials

Colab already includes NumPy, Pillow, and matplotlib. The first cell installs the OpenAI-compatible client and Meta's SAM response parser. Enter keys in Colab Secrets, or use the hidden prompts below.

In [ ]:
%pip -q install openai meta-sam-parser

In [ ]:
import base64
import io
import json
import os
from dataclasses import dataclass
from getpass import getpass

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps
from openai import AsyncOpenAI, OpenAI

try:
    from google.colab import files, userdata
except ImportError as exc:
    raise RuntimeError("This notebook expects Google Colab for its upload widget.") from exc

def secret(name, required=False):
    value = os.environ.get(name)
    if not value:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value and required:
        value = getpass(f"{name}: ")
    if value:
        os.environ[name] = value
    return value

BASETEN_API_KEY = secret("BASETEN_API_KEY", required=True)
META_API_KEY = secret("MODEL_API_KEY")  # Needed only if SAM runs.
if not BASETEN_API_KEY:
    raise ValueError("BASETEN_API_KEY is required. Add it to Colab Secrets.")

BASETEN_MODEL = "meta-models/muse-spark-open-1026"
BASETEN_BASE_URL = "https://inference.baseten.co/v1"
SAM_MODEL = "sam-3.1"
MAX_SEPARATE_PLANTS = 8
MAX_SEGMENTS = 6
USE_JSON_SCHEMA_RESPONSE_FORMAT = False  # Switch on only if this endpoint supports it.
baseten_client = OpenAI(base_url=BASETEN_BASE_URL, api_key=os.environ["BASETEN_API_KEY"], timeout=120.0, max_retries=0)
print("Baseten model:", BASETEN_MODEL)
print("SAM available:", bool(META_API_KEY))

## 2. Upload and normalize one photo

Use the same accepted formats, single-frame check, EXIF orientation, 2048-pixel maximum side, and JPEG quality as the deployed API. The source file stays unchanged.

In [ ]:
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError(f"Upload exactly one image; received {len(uploaded)} files.")
image_name, original_bytes = next(iter(uploaded.items()))
if len(original_bytes) > 12 * 1024 * 1024:
    raise ValueError("Image exceeds the app's 12 MB upload limit.")
with Image.open(io.BytesIO(original_bytes)) as opened:
    if opened.format not in {"JPEG", "PNG", "WEBP"} or getattr(opened, "n_frames", 1) != 1:
        raise ValueError("Upload a single-frame JPEG, PNG, or WebP image.")
    image = ImageOps.exif_transpose(opened).convert("RGB")
if min(image.size) < 2:
    raise ValueError("Image dimensions must be at least 2 × 2 pixels.")
image.thumbnail((2048, 2048), Image.Resampling.LANCZOS)
buffer = io.BytesIO()
image.save(buffer, format="JPEG", quality=92, optimize=True)
image_bytes = buffer.getvalue()

def data_url(content, mime="image/jpeg"):
    return f"data:{mime};base64," + base64.b64encode(content).decode("ascii")

print(f"{image_name}: {image.width} × {image.height}")
plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.axis("off");

## 3. Assess the whole photo with the deployed prompt

The prompt and allowed lookup keys below are embedded snapshots of the project. Baseten's hackathon endpoint may or may not support strict `response_format=json_schema`; the default sends the schema as an instruction and validates the returned JSON locally. If you confirm schema mode works, set `USE_JSON_SCHEMA_RESPONSE_FORMAT=True` above. A malformed or truncated response is shown as an error rather than treated as a valid finding.

In [ ]:
PLANT_ASSESSMENT_PROMPT = '\nYou are a visual plant investigation assistant.\n\nAnalyze the provided photograph to identify visually distinct\nplant types, determine whether segmentation would improve\nidentification, and decide whether another photograph is needed.\n\nBOUQUET DETECTION\n\n- Determine whether the photograph contains a bouquet of cut flowers.\n- A bouquet may contain one or multiple plant species.\n- Do not classify potted plants or naturally growing flower\n  clusters as bouquets.\n- Set bouquet_type to single_species, mixed_species, or uncertain.\n- Use not_bouquet when is_bouquet is false.\n- Do not assume all flowers in a bouquet belong to one species.\n\nDISTINCT PLANT DETECTION\n\n- First inventory the whole frame before naming any plants: inspect the\n  foreground, center, background, and gaps between prominent blooms.\n- In mixed bouquets, look specifically for small filler flowers, partially\n  covered blooms, and foliage that may belong to a different plant type.\n- Recheck the inventory after identifying the dominant flowers so a small\n  but visibly different type is not omitted.\n- Identify every visually distinct plant type in the photograph.\n- Count distinct plant types, NOT individual flowers, leaves,\n  stems, or blossoms.\n- Multiple flowers belonging to the same apparent plant type\n  count as ONE group.\n- Several visually identical plants may belong to one group.\n- Separate visually different plant types into different groups.\n- Include distinct unidentified plant types as Unknown.\n- If a visible flower is clearly different but its identity is uncertain,\n  include an Unknown group and recommend inspection when useful.\n- Do not count different parts of the same plant as separate types.\n- Do not invent plants hidden behind visible objects.\n- Set distinct_plant_count equal to the number of plant_groups.\n- Use 0 and an empty plant_groups array when no plants are visible.\n\nSEPARATE PLANT COUNT\n\n- Set separate_plant_count to the number of clearly separate whole plants\n  visible outside a bouquet. Count each distinct potted plant once, even\n  when several pots contain the same species.\n- Do not count individual flowers, flower heads, leaves, or stems as plants.\n- For a bouquet of cut flowers, set separate_plant_count to 0.\n- When separate_plant_count exceeds {max_separate_plants}, request a closer photograph of\n  one plant at a time and do not recommend SAM for the whole scene.\n\nPLANT IDENTIFICATION\n\nFor each visually distinct plant group:\n\n- Provide the most likely common name.\n- Provide the scientific name when supported by visual evidence.\n- Prefer genus-level identification when species is uncertain.\n- Use Unknown when identification is unreliable.\n- Describe distinguishing visible features.\n- Assign qualitative identification confidence: high, medium, or low.\n- Set needs_inspection when further examination could\n  materially improve identification.\n\nDo not merge different plant species merely because they\nhave similar colors or appear in the same arrangement.\n\nDo not invent confidence percentages.\n\nTOXICITY DATABASE MATCHING\n\n- Propose a toxicity_lookup_key for each plant group.\n- Match the identified taxon to an allowed database key.\n- Use unknown when no reliable matching key exists.\n- Do not substitute a different plant to obtain a database match.\n- Do not infer toxicity or pet safety.\n- Do not provide veterinary or medical advice.\n- The application independently verifies all database matches.\n\nSEGMENTATION DECISION\n\nDetermine whether object segmentation would improve\nidentification or investigation.\n\nRecommend segmentation when:\n\n- Multiple visually distinct plant types overlap.\n- Different species may be present but are difficult to separate.\n- Background clutter interferes with identification.\n- A plant region is partially obscured.\n- An unidentified plant group requires closer inspection.\n- Isolating individual regions could reveal useful\n  identifying characteristics.\n\nDo not recommend segmentation solely because several\nflowers of the same plant type are visible.\n\nSegmentation cannot recover details hidden from the camera.\n\nExplain briefly why segmentation would or would not help.\n\nSEGMENTATION TARGET\n\nChoose the SAM target that would provide the most useful isolated\nregions if segmentation runs. Return exactly one of:\n\n- flower: segment individual flower heads when blooms from different\n  apparent plant types overlap and the heads contain the useful\n  identifying detail.\n- plant: segment whole plants when leaves, stems, growth habit, or\n  the relationship between flowers and foliage matters; prefer this\n  for several potted plants or flowers that look too similar alone.\n\nChoose the best target even when suggest_segmentation is false so an\nexplicit user request to run SAM has a target. This is a visual\nrecommendation, not a claim that SAM has already found a mask.\nExplain the target choice briefly in segmentation_target_reason,\nseparately from the decision to run segmentation.\n\nADDITIONAL PHOTO DECISION\n\nDetermine whether another photograph would materially improve\nplant identification.\n\nRequest another photograph when:\n\n- Important identifying characteristics are missing.\n- A plant is partially hidden.\n- The image is blurry, dark, or too distant.\n- Similar-looking species cannot be distinguished.\n- Multiple plants cannot be reliably separated.\n- Identification remains uncertain.\n- More than {max_separate_plants} separate whole plants are visible outside a bouquet.\n\nWhen requesting another photograph:\n\n- Identify the specific plant or region requiring attention.\n- Specify exactly what the user should photograph.\n- Request the most useful angle or botanical feature.\n- Prefer views showing flowers, leaves, stems, or the entire plant.\n- Give clear, actionable instructions.\n- Explain briefly why the requested photograph would help.\n- Request only ONE additional photograph at a time.\n\nExamples:\n\n"Photograph the entire plant, including its leaves and stem."\n"Take a close-up of the white flower on the left."\n"Photograph where the leaves attach to the stem."\n"Move closer to the unidentified plant behind the roses."\n"Take a sharper photograph in natural daylight."\n\nIf another photograph is unnecessary:\n- Set another_photo_needed to false.\n- Return empty strings for photo_instructions and photo_reason.\n\nIMPORTANT RULES\n\n- Analyze only visible evidence.\n- Never assume every plant has been identified.\n- Unknown plant groups must remain unresolved.\n- Do not infer toxicity or declare plants safe for pets.\n- Identification confidence is qualitative, not calibrated.\n- Return only valid JSON matching the provided schema.\n'
ALLOWED_KEYS = 'aglaonema, aloe vera, chamaedorea elegans, chlorophytum comosum, chrysanthemum, crassula ovata, cyclamen, dieffenbachia, epipremnum aureum, euphorbia pulcherrima, hemerocallis, monstera deliciosa, narcissus, nephrolepis exaltata, phalaenopsis, philodendron hederaceum, saintpaulia, spathiphyllum, alocasia, amaryllis, hippeastrum, anthurium, arrowhead vine, azalea, bay laurel, begonia, caladium, calla lily, carnation, castor bean, catnip, clivia lily, corn plant, dracaena marginata, english ivy, eucalyptus, foxglove, gardenia, geranium, gladiolus, holly, hyacinth, hydrangea, kalanchoe, lily, nandina, oxalis, pencil cactus, sago palm, snake plant, tulip, wisteria, yucca, asparagus fern, bird of paradise flower, christmas rose, areca palm, aluminum plant, artillery plant, polka dot plant, ball fern, baby rubber plant, basil, burros tail, calathea, camellia, cast iron plant, christmas cactus, creeping charlie, crepe myrtle, friendship plant, freesia, hens and chicks, hibiscus, hoya, impatiens, jasmine, kentia palm, money tree, norfolk island pine, orchid, petunia, pony tail palm, rose, rosemary, sage, snapdragon, sunflower, zinnia, cape primrose, goldfish plant, nerve plant, purple waffle plant, rubber plant, zz plant, pilea peperomioides, monstera adansonii, croton, lavender, ficus lyrata, marigold'
ASSESSMENT_SCHEMA = json.loads('{"type":"object","additionalProperties":false,"properties":{"is_bouquet":{"type":"boolean"},"bouquet_type":{"type":"string","enum":["single_species","mixed_species","uncertain","not_bouquet"]},"distinct_plant_count":{"type":"integer"},"separate_plant_count":{"type":"integer"},"plant_groups":{"type":"array","items":{"type":"object","additionalProperties":false,"properties":{"id":{"type":"integer"},"plant_identification":{"type":"string"},"scientific_name":{"type":"string"},"toxicity_lookup_key":{"type":"string"},"identification_confidence":{"type":"string","enum":["high","medium","low"]},"visual_evidence":{"type":"array","items":{"type":"string"}},"needs_inspection":{"type":"boolean"}},"required":["id","plant_identification","scientific_name","toxicity_lookup_key","identification_confidence","visual_evidence","needs_inspection"]}},"suggest_segmentation":{"type":"boolean"},"segmentation_target":{"type":"string","enum":["flower","plant"]},"segmentation_target_reason":{"type":"string"},"segmentation_reason":{"type":"string"},"another_photo_needed":{"type":"boolean"},"photo_instructions":{"type":"string"},"photo_reason":{"type":"string"}},"required":["is_bouquet","bouquet_type","distinct_plant_count","separate_plant_count","plant_groups","suggest_segmentation","segmentation_target","segmentation_target_reason","segmentation_reason","another_photo_needed","photo_instructions","photo_reason"]}')

In [ ]:
DEVELOPER_PROMPT = (
    PLANT_ASSESSMENT_PROMPT.format(max_separate_plants=MAX_SEPARATE_PLANTS)
    + "\nAllowed toxicity_lookup_key values: " + ALLOWED_KEYS + ", unknown."
    + "\nReturn a JSON object with this schema: " + json.dumps(ASSESSMENT_SCHEMA, separators=(",", ":"))
)
REQUIRED = set(ASSESSMENT_SCHEMA["required"])

def parse_assessment(content):
    if not content:
        raise ValueError("Baseten returned empty content.")
    text = content.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    result = json.loads(text)
    missing = REQUIRED - result.keys()
    if missing:
        raise ValueError(f"Assessment is missing fields: {sorted(missing)}")
    if result["distinct_plant_count"] != len(result["plant_groups"]):
        print("Warning: distinct_plant_count differs from plant_groups length.")
    return result

def assess(image_content, mime="image/jpeg", extra_instruction="Assess this photograph."):
    kwargs = dict(
        model=BASETEN_MODEL,
        messages=[
            {"role": "developer", "content": DEVELOPER_PROMPT},
            {"role": "user", "content": [
                {"type": "text", "text": extra_instruction},
                {"type": "image_url", "image_url": {"url": data_url(image_content, mime)}},
            ]},
        ],
        max_tokens=4000,
    )
    if USE_JSON_SCHEMA_RESPONSE_FORMAT:
        kwargs["response_format"] = {
            "type": "json_schema",
            "json_schema": {"name": "plant_image_assessment", "strict": True, "schema": ASSESSMENT_SCHEMA},
        }
    response = baseten_client.chat.completions.create(**kwargs)
    choice = response.choices[0]
    if choice.finish_reason == "length":
        raise RuntimeError("Baseten truncated the response; increase max_tokens.")
    raw = choice.message.content
    try:
        result = parse_assessment(raw)
    except (ValueError, json.JSONDecodeError) as exc:
        print("Unparsed Baseten response:", raw)
        raise RuntimeError("Baseten did not return a valid assessment JSON object.") from exc
    usage = response.usage
    if usage:
        print("Tokens · input:", usage.prompt_tokens, "output:", usage.completion_tokens)
    return result

scene = assess(image_bytes)
print(json.dumps(scene, indent=2, ensure_ascii=False))

## 4. Apply the app's segmentation decision

SAM is skipped when the scene reports more than eight separate whole plants. Otherwise the app requests SAM if Muse suggests it, reports several plant types, marks the bouquet mixed or uncertain, or has a low-confidence group needing inspection. You can change `RUN_SAM` to `False` to compare scene-only results. A Baseten key does **not** authorize Meta SAM calls.

In [ ]:
def should_segment(assessment):
    if assessment["separate_plant_count"] > MAX_SEPARATE_PLANTS:
        return False, "Too many separate plants; request individual photos."
    reasons = []
    if assessment["suggest_segmentation"]:
        reasons.append("Model recommended region inspection")
    if assessment["distinct_plant_count"] > 1:
        reasons.append("Multiple plant types reported")
    if assessment["bouquet_type"] in {"mixed_species", "uncertain"}:
        reasons.append("Mixed or uncertain bouquet")
    if any(g["needs_inspection"] and g["identification_confidence"] == "low"
           for g in assessment["plant_groups"]):
        reasons.append("Low-confidence group needs inspection")
    return bool(reasons), "; ".join(reasons) or "No segmentation trigger"

RUN_SAM, segmentation_reason = should_segment(scene)
SAM_PROMPT = scene["segmentation_target"]  # flower or plant, as chosen by the model
print("Run SAM:", RUN_SAM, "· Target:", SAM_PROMPT, "·", segmentation_reason)
if RUN_SAM and not META_API_KEY:
    print("Add MODEL_API_KEY to Colab Secrets, rerun the credential cell, then run SAM.")

## 5. SAM 3.1: select useful masks and inspect crops

This cell uses the Meta Model API, the `one_bit` mask encoding, and the same area, size, near-duplicate, and diversity policy as the deployed app. It shows real selected masks and transparent crops. No SAM call is made when `RUN_SAM` is false or the Meta key is absent.

In [ ]:
from meta_sam_parser import decode_mask_to_raster, image_segmentation_format, parse_responses_stream

@dataclass(eq=False)
class MaskCandidate:
    object_id: str
    bounds: tuple[int, int, int, int]
    mask: np.ndarray
    pixels: int
    confidence: float | None

def mask_candidate(record):
    local = np.frombuffer(decode_mask_to_raster(record.mask), dtype=np.uint8)
    local = local.reshape(record.mask.height, record.mask.width)
    b = record.bounds
    left, top = max(0, int(np.floor(b.left))), max(0, int(np.floor(b.top)))
    right, bottom = min(image.width, int(np.ceil(b.right))), min(image.height, int(np.ceil(b.bottom)))
    if right <= left or bottom <= top:
        return None
    local_mask = Image.fromarray(local * 255).resize((right-left, bottom-top), Image.Resampling.NEAREST)
    mask = np.asarray(local_mask) > 0
    ys, xs = np.where(mask)
    pixels = len(xs)
    if pixels < max(300, int(image.width * image.height * 0.001)) or pixels > image.width * image.height * 0.9:
        return None
    if xs.max()-xs.min()+1 < 32 or ys.max()-ys.min()+1 < 32:
        return None
    crop_box = (int(xs.min()), int(ys.min()), int(xs.max())+1, int(ys.max())+1)
    tight = (left+crop_box[0], top+crop_box[1], left+crop_box[2], top+crop_box[3])
    raw = getattr(record, "confidence", None)
    try:
        confidence = float(raw) if raw is not None else None
    except (TypeError, ValueError):
        confidence = None
    if confidence is not None and not np.isfinite(confidence):
        confidence = None
    return MaskCandidate(str(record.object_id), tight,
        mask[crop_box[1]:crop_box[3], crop_box[0]:crop_box[2]], pixels, confidence)

def intersection(a, b):
    left, top = max(a.bounds[0],b.bounds[0]), max(a.bounds[1],b.bounds[1])
    right, bottom = min(a.bounds[2],b.bounds[2]), min(a.bounds[3],b.bounds[3])
    if right <= left or bottom <= top:
        return 0
    am = a.mask[top-a.bounds[1]:bottom-a.bounds[1], left-a.bounds[0]:right-a.bounds[0]]
    bm = b.mask[top-b.bounds[1]:bottom-b.bounds[1], left-b.bounds[0]:right-b.bounds[0]]
    return int(np.count_nonzero(am & bm))

def quality(c):
    box_area = (c.bounds[2]-c.bounds[0]) * (c.bounds[3]-c.bounds[1])
    size = min(1.0, (c.pixels / max(1, image.width*image.height*0.02))**0.5)
    fill = c.pixels / max(1, box_area)
    confidence = max(0.0,min(1.0,c.confidence)) if c.confidence is not None else 0.5
    return 0.5*size + 0.2*fill + 0.3*confidence

def select_masks(candidates):
    unique = []
    for c in sorted(candidates, key=quality, reverse=True):
        duplicate = False
        for kept in unique:
            overlap = intersection(c, kept)
            union = c.pixels + kept.pixels - overlap
            ratio = min(c.pixels,kept.pixels)/max(c.pixels,kept.pixels)
            if overlap/max(1,union) >= 0.8 or (ratio >= 0.65 and overlap/max(1,min(c.pixels,kept.pixels)) >= 0.92):
                duplicate = True
                break
        if not duplicate:
            unique.append(c)
    selected = []
    while unique and len(selected) < MAX_SEGMENTS:
        best = max(unique, key=lambda c: quality(c) + 0.5*(1-max(
            (intersection(c,p)/c.pixels for p in selected), default=0)))
        selected.append(best)
        unique.remove(best)
    return selected

async def call_sam():
    client = AsyncOpenAI(base_url="https://api.meta.ai/v1", api_key=os.environ["MODEL_API_KEY"],
                         timeout=180.0, max_retries=0)
    try:
        events = await client.responses.create(
            model=SAM_MODEL,
            input=[{"type":"message","role":"user","content":[
                {"type":"input_text","text":SAM_PROMPT},
                {"type":"input_image","image_url":data_url(image_bytes)},
            ]}],
            stream=True,
            metadata={"mask_encoding":"one_bit","include_confidence":"true"},
        )
        return await parse_responses_stream(events, image_segmentation_format()).final_result()
    finally:
        await client.close()

selected = []
crops = []
if RUN_SAM and META_API_KEY:
    sam_result = await call_sam()
    records = [r for r in sam_result.records if getattr(r, "kind", None) == "mask"]
    selected = select_masks([c for r in records if (c := mask_candidate(r)) is not None])
    for c in selected:
        rgba = np.asarray(image.crop(c.bounds).convert("RGBA")).copy()
        rgba[:,:,3] = c.mask.astype(np.uint8)*255
        buf = io.BytesIO()
        Image.fromarray(rgba).save(buf, format="PNG", optimize=True)
        crops.append({"object_id":c.object_id,"bounds":c.bounds,"image":Image.fromarray(rgba),"bytes":buf.getvalue()})
    print(f"SAM masks: {len(records)} · filtered: {len(records)-len(selected)} · selected: {len(crops)}")
    if sam_result.diagnostics:
        print("Parser diagnostics:", sam_result.diagnostics)
    if crops:
        fig, axes = plt.subplots(1, len(crops), figsize=(3*len(crops), 4), squeeze=False)
        for ax, crop in zip(axes.flat, crops):
            ax.imshow(crop["image"])
            ax.set_title(f"Region {crop['object_id']}")
            ax.axis("off")
        plt.tight_layout()
else:
    print("SAM skipped.")

## 6. Reassess selected regions on Baseten

Each crop goes to the **same Baseten model ID and deployed scene prompt**. The region instruction matches the app: inspect only the visible crop, preserve uncertainty, and leave follow-up decisions to the whole-scene result. Compare names, group counts, uncertainty, and segmentation choices; region results are not independent ground truth.

In [ ]:
REGION_INSTRUCTION = (
    "This is a transparent-background crop of a SAM-selected plant region. "
    "Assess only visible plant types in this crop and preserve uncertainty. "
    "The crop omits scene context, so do not request another photograph "
    "or infer where this plant is relative to other plants. "
    "The whole-scene assessment decides whether a follow-up photo is needed."
)
region_results = []
for crop in crops:
    result = assess(crop["bytes"], "image/png", REGION_INSTRUCTION)
    result["another_photo_needed"] = False
    result["photo_instructions"] = ""
    result["photo_reason"] = ""
    region_results.append({"object_id": crop["object_id"], "assessment": result})
    print(f"Region {crop['object_id']}: {[g['plant_identification'] for g in result['plant_groups']]}")

print("Whole image:", [g["plant_identification"] for g in scene["plant_groups"]])
print("Selected regions reassessed:", len(region_results))

## 7. Compare with a deployed Muse Spark result

If you saved the deployed app's **whole-scene JSON** for the same photo, paste it into `DEPLOYED_SCENE_JSON` below. Leave it empty to inspect only Baseten's result. Differences can come from model weights, serving, decoding, image normalization, SAM masks, or prompt/schema enforcement. This notebook makes no claim that either output is correct without image-level review.

In [ ]:
DEPLOYED_SCENE_JSON = """"""  # Optional: paste only the whole-scene assessment JSON.
if DEPLOYED_SCENE_JSON.strip():
    deployed = json.loads(DEPLOYED_SCENE_JSON)
    rows = [
        ("Distinct plant types", scene["distinct_plant_count"], deployed.get("distinct_plant_count")),
        ("Plant names", [g["plant_identification"] for g in scene["plant_groups"]],
         [g["plant_identification"] for g in deployed.get("plant_groups", [])]),
        ("Needs another photo", scene["another_photo_needed"], deployed.get("another_photo_needed")),
        ("Suggests segmentation", scene["suggest_segmentation"], deployed.get("suggest_segmentation")),
        ("SAM target", scene["segmentation_target"], deployed.get("segmentation_target")),
    ]
    for label, baseten, deployed_value in rows:
        print(f"{label}:\n  Baseten: {baseten}\n  Deployed: {deployed_value}")
else:
    print("No deployed assessment supplied. Baseten scene and regional findings are shown above.")

## Notes to record after running

- **Visible plant types missed or added:** _fill after inspection_
- **Did SAM isolate useful regions?** _fill after inspection_
- **Did regional reassessment change an identification?** _fill after inspection_
- **Follow-up photo decision:** _fill after inspection_

These are visual identification experiments. The notebook does not perform the local source-backed cat/dog toxicity lookup. Do not treat either model's plant name as a safety determination.